# EvoGP tutorial: symbolic regression on the torch-free `evogp.gp` stack

`evogp.gp` is the functional genetic-programming stack of EvoGP 2.0.0a2. It is built on **`evoxc`** (EvoX Compiler) graphs plus **numpy** host data: one generation is one traced `evoxc` graph over a `[pop_size, max_nodes]` prefix-order channel triple, and no module of the stack imports torch, triton or a device runtime.

Two properties run through the whole notebook:

* **Determinism is key-based.** The RNG stream is an `evoxc` splitmix64 key carried *inside* the state, so `run(..., key=0)` reproduces a run exactly -- there is no global seed and nothing to reset.
* **The fitness contract is one line.** `evaluate(channels)` returns a `(pop_size,)` float32 vector that is **maximized** -- a loss is negated, and a non-finite loss becomes `-inf`.

Every cell below runs on a CPU box; the last section adds the optional device path.

In [1]:
import time

import numpy as np

from evogp.gp import (
    GPConfig,
    GPTreeConfig,
    SymbolicRegression,
    init_gp,
    row_length,
    run,
)

# The tree layer's own id/name tables. evogp.gp has no public expression
# exporter yet (MIGRATION.md step 9), so this notebook -- like
# example/gp_sr.py -- reads the private module as the stand-in.
from evogp.gp._constants import FUNC_ARITY, FUNCS_NAMES, Func, NType

## The task: `x0**2 + x1**2`

A synthetic symbolic-regression problem on 800 rows drawn uniformly from `[-5, 5]`, the same problem as `example/gp_sr.py`. The target is reachable with the shipped `+ - * /` function set and a constant table, so the run converges onto it instead of merely improving, which keeps the printed expression short and checkable.

The data is plain numpy: `x` is `[n_rows, 2]` float32 and `y` is `[n_rows]` float32. The draw uses its own `np.random.default_rng(seed)`, independent of the driver's key.

In [2]:
NUM_DATA = 800
INPUT_LEN = 2
DATA_BOUNDS = (-5.0, 5.0)
SEED = 0


def target_function(x):
    """`x0**2 + x1**2` for `x` of shape `[n_rows, 2]`."""
    values = np.asarray(x, dtype=np.float64)
    return values[:, 0] ** 2 + values[:, 1] ** 2


def make_data(num_data=NUM_DATA, seed=SEED):
    """The synthetic problem: `(x [num_data, 2], y [num_data])` float32."""
    generator = np.random.default_rng(seed)
    low, high = DATA_BOUNDS
    x = generator.uniform(low, high, size=(int(num_data), INPUT_LEN))
    x = x.astype(np.float32)
    return x, target_function(x).astype(np.float32)


x, y = make_data()
problem = SymbolicRegression(x, y)
print(f"x {x.shape} {x.dtype}   y {y.shape} {y.dtype}")
for row in range(3):
    first, second = float(x[row, 0]), float(x[row, 1])
    print(f"  x0 {first:+7.3f}  x1 {second:+7.3f}  ->  target {float(y[row]):+7.3f}")

x (800, 2) float32   y (800,) float32
  x0  +1.370  x1  -2.302  ->  target  +7.176
  x0  -4.590  x1  -4.835  ->  target +44.445
  x0  +3.133  x1  +4.128  ->  target +26.851


## Describing the population: `GPTreeConfig` + `GPConfig`

Both are frozen dataclasses holding **static leaves only** (Python scalars and tuples). Nothing is mutated and nothing is traced, which is what lets the encoding layer bake a config into one compiled graph.

* `GPTreeConfig` is the tree descriptor -- the old `GenerateDescriptor`: `max_nodes` (row width, formerly `max_tree_len`), `input_len`, the `using_funcs` function set (names, or `(name, weight)` pairs), the depth schedule (`max_layer_cnt`, `layer_leaf_prob`), and the leaf mix (`const_prob`, `const_range`, `sample_cnt`). Its derived tables (`roulette_funcs`, `func_arity`, `depth2leaf_probs`, `max_arity`, `levels`) are computed once in `__post_init__`.
* `GPConfig` is the population and the operator rates: `pop_size`, `elite_count` (the best N copied unchanged into the next generation), `tournament_size`, `crossover_prob`, `mutation_prob`, `mutation_kind`.

In [3]:
POP_SIZE = 256
GENERATIONS = 30
MAX_NODES = 64
USING_FUNCS = ("+", "-", "*", "/")
MAX_LAYER_CNT = 4
LAYER_LEAF_PROB = 0.2
CONST_PROB = 0.5
CONST_RANGE = (-5.0, 5.0)
SAMPLE_CNT = 8

tree_config = GPTreeConfig(
    max_nodes=MAX_NODES,
    input_len=INPUT_LEN,
    using_funcs=USING_FUNCS,
    max_layer_cnt=MAX_LAYER_CNT,
    layer_leaf_prob=LAYER_LEAF_PROB,
    const_prob=CONST_PROB,
    const_range=CONST_RANGE,
    sample_cnt=SAMPLE_CNT,
)
gp_config = GPConfig(
    pop_size=POP_SIZE,
    elite_count=2,
    tournament_size=4,
    crossover_prob=0.9,
    mutation_prob=0.2,
    mutation_kind="subtree",
)
config = (tree_config, gp_config)

print(f"function set   {USING_FUNCS} -- max arity {tree_config.max_arity}, "
      f"{tree_config.levels} depth levels")
print(f"leaf mix       const_prob {tree_config.const_prob}, "
      f"{len(tree_config.const_samples)} constants in {tree_config.const_range}")
print(f"depth schedule {tree_config.depth2leaf_probs}")
print(gp_config)

function set   ('+', '-', '*', '/') -- max arity 2, 4 depth levels
leaf mix       const_prob 0.5, 8 constants in (-5.0, 5.0)
depth schedule (0.2, 0.2, 0.2, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0)
GPConfig(pop_size=256, elite_count=2, tournament_size=4, crossover_prob=0.9, mutation_prob=0.2, selection='tournament', selection_pressure=0.5, mutation_kind='subtree')


## `init_gp` and `run`: the whole loop

The driver is three plain functions -- `init_gp(config, key) -> state`, `step(config, state, evaluate) -> state`, and `run(config, evaluate, generations=..., key=...) -> GPResult` -- and `evaluate` is the entire plug-in surface. `SymbolicRegression(x, y).evaluate` is the numpy host evaluator: fitness is `-mean((y - pred)**2)`, so an exact fit reads `0.0`.

Elitism is the driver's job (`GPConfig.elite_count`), which is what makes the per-generation best fitness monotone non-decreasing. `run` also takes a `callback(generation, state, fitness)`; that is the reporting hook used below.

In [4]:
state = init_gp(config, SEED)
print(f"generation 0   {POP_SIZE} trees, channel width {MAX_NODES}, "
      f"fitness vector {tuple(state.fitness.shape)}, RNG key in the state")


def report(generation, _state, fitness):
    """Print generation 0, every fifth one, and the last."""
    if generation % 5 and generation != GENERATIONS - 1:
        return
    finite = fitness[np.isfinite(fitness)]
    print(f"generation {generation:3d}  best {float(np.max(fitness)):+12.5f}  "
          f"median {float(np.median(finite)):+12.5f}")


started = time.perf_counter()
result = run(config, problem.evaluate, generations=GENERATIONS, key=SEED,
             callback=report)
wall = time.perf_counter() - started

history = result.best_fitness_history
print(f"\n{POP_SIZE} trees x {GENERATIONS} generations over {NUM_DATA} rows in "
      f"{wall:.2f}s ({1000.0 * wall / GENERATIONS:.1f} ms/generation)")
print(f"best fitness   {result.best_fitness:+.5f} at generation "
      f"{result.best_generation} of {len(history)} evaluated")
print(f"fitness path   {history[0]:+.5f} -> {history[-1]:+.5f}  "
      f"(monotone: {all(a <= b for a, b in zip(history, history[1:]))})")

generation 0   256 trees, channel width 64, fitness vector (256,), RNG key in the state
generation   0  best   -127.47717  median   -589.98407
generation   5  best    -63.56912  median   -238.92290
generation  10  best    -56.53069  median   -143.08543
generation  15  best    -56.53069  median   -144.99811
generation  20  best    -12.82594  median    -57.50838
generation  25  best     -0.00000  median    -78.57141
generation  29  best     -0.00000  median    -85.18500

256 trees x 30 generations over 800 rows in 1.05s (35.0 ms/generation)
best fitness   -0.00000 at generation 28 of 30 evaluated
fitness path   -127.47717 -> -0.00000  (monotone: True)


## Determinism: the same key, the same numbers

`run(..., key=0)` seeds the generation-0 population and every operator draw from that one integer, and the key travels in the state. Run it a second time and every fitness number is identical; only the timings differ. That is the whole reproducibility contract of the stack, and it is how the library's own tests and reports compare runs.

In [5]:
second = run(config, problem.evaluate, generations=GENERATIONS, key=SEED)

print(f"first run    best {result.best_fitness:+.6e}")
print(f"second run   best {second.best_fitness:+.6e}")
print(f"identical    best value   {second.best_fitness == result.best_fitness}")
print(f"identical    history      "
      f"{second.best_fitness_history == result.best_fitness_history}")
winner_equal = all(
    np.array_equal(a, b)
    for a, b in zip(result.best_channels, second.best_channels)
)
print(f"identical    winner rows  {winner_equal}")

first run    best -1.049584e-12
second run   best -1.049584e-12
identical    best value   True
identical    history      True
identical    winner rows  True


## Reading the winner: an infix expression

`GPResult.best_channels` is the winning tree as host `(opcode, payload, size)` rows in prefix order. Hand the triple back to `problem.evaluate` or `problem.predict` and it reads as a population of one.

`evogp.gp` has **no public expression exporter yet** (`MIGRATION.md` step 9: the torch stack's `to_sympy_expr` went away with its stack), so the renderer below is the same local decoder `example/gp_sr.py` uses. A `size[i] == 1` cell is a leaf -- `NType.CONST` renders the payload as a number, `NType.VAR` renders `x{int(payload)}` -- and a function node's children start at `i + 1`, each next child following the previous child's own `size`.

In [6]:
def render_expression(channels, input_len=INPUT_LEN):
    """Readable infix string of one tree (the stand-in for `to_sympy_expr`)."""
    opcode, payload, size = (np.asarray(part) for part in channels)
    width = int(size.shape[0])

    def walk(node, depth):
        """Render the subtree rooted at `node` (`depth` bounds the walk)."""
        if node >= width or size[node] <= 0 or depth > width:
            return "?"
        if size[node] == 1:
            if int(opcode[node]) == NType.CONST:
                return f"{float(payload[node]):.4g}"
            return f"x{int(float(payload[node])) % input_len}"
        function_id = int(opcode[node])
        children, child = [], node + 1
        for _ in range(FUNC_ARITY[function_id]):
            if child >= width:
                children.append("?")
                break
            children.append(walk(child, depth + 1))
            child += int(size[child])
        return _apply(function_id, children)

    return walk(0, 0)


def _apply(function_id, children):
    """Render one function node from its already-rendered children."""
    if len(children) == 3:
        return f"if({children[0]}, {children[1]}, {children[2]})"
    if len(children) == 2:
        symbol = {Func.ADD: "+", Func.SUB: "-", Func.MUL: "*", Func.DIV: "/"}
        if function_id in symbol:
            return f"({children[0]} {symbol[function_id]} {children[1]})"
        return f"{FUNCS_NAMES[function_id]}({children[0]}, {children[1]})"
    if len(children) == 1:
        if function_id == Func.NEG:
            return f"(-{children[0]})"
        return f"{FUNCS_NAMES[function_id]}({children[0]})"
    return FUNCS_NAMES[function_id]


print(f"winning tree   {int(row_length(result.best_channels)[0])} nodes")
print(f"expression     {render_expression(result.best_channels)}")

predictions = problem.predict(result.best_channels)[0]
rmse = float(np.sqrt(np.mean((predictions - y) ** 2, dtype=np.float64)))
print(f"rmse           {rmse:.5f}")
for row in range(2):
    print(f"  row {row}          prediction {predictions[row]:+12.6f}  "
          f"target {y[row]:+12.6f}")

winning tree   23 nodes
expression     ((((x1 * ((x1 + 4.128) - 4.128)) + (x0 * x0)) - (4.128 + 1.37)) + ((x0 - x0) + (4.128 + 1.37)))
rmse           0.00000
  row 0          prediction    +7.175666  target    +7.175666
  row 1          prediction   +44.445084  target   +44.445084


## Optional: the device path

The same driver runs device-resident. Nothing above changes except two keyword arguments -- `init_gp(..., backend="iree", device=kind)` and `run(..., backend="iree", device=kind)` -- plus the objective: `GraphEvaluator` is the torch-free in-graph evaluator (one `evoxc` graph per shape, the fitness stays a device tensor, zero readbacks per generation), and `FusedDeviceEvaluator` is the same contract served by the sibling Triton kernel, chosen when `fused_available()` and `interop_available(kind)` both answer True.

The cell is written as a probe, not an assumption: with no `iree` packages or no rocm/cuda device the probe answers `None` and the run falls back to the numpy host loop, and a device-route failure prints its reason and takes the same fallback -- so the notebook still executes end to end on a CPU box. The device population here is deliberately its own, smaller one with `max_nodes=32`: the in-graph evaluator at `max_nodes=64` is currently declined by the iree lowering of the channel graph, while 32 compiles (a lowering limitation, not a stack limit).

In [7]:
import importlib.util

from evogp.gp import (
    FusedDeviceEvaluator,
    GraphEvaluator,
    fused_available,
    interop_available,
)

DEVICE_CONFIG = (
    GPTreeConfig(max_nodes=32, input_len=INPUT_LEN, using_funcs=USING_FUNCS,
                 max_layer_cnt=MAX_LAYER_CNT, layer_leaf_prob=LAYER_LEAF_PROB,
                 const_prob=CONST_PROB, const_range=CONST_RANGE,
                 sample_cnt=SAMPLE_CNT),
    GPConfig(pop_size=128, elite_count=2, tournament_size=4,
             crossover_prob=0.9, mutation_prob=0.2, mutation_kind="subtree"),
)


def device_kind():
    """`"rocm"` / `"cuda"` when an iree device is usable here, else `None`."""
    if importlib.util.find_spec("iree.compiler") is None:
        return None
    if importlib.util.find_spec("iree.runtime") is None:
        return None
    try:
        import evoxc

        devices = evoxc.devices()
    except Exception:  # noqa: BLE001 - a probe answers "no device", never raises
        return None
    for kind in ("rocm", "cuda"):
        if any(device.kind == kind for device in devices):
            return kind
    return None


def run_device(config, x, y, generations=5):
    """The device driver, or the numpy host loop when no device route works."""
    kind = device_kind()
    if kind is not None:
        try:
            if fused_available() and interop_available(kind):
                route = "fused kernel"
                evaluate = FusedDeviceEvaluator(x, y)
            else:
                route = "in-graph iree"
                evaluate = GraphEvaluator(x, y, used_ids=config[0],
                                          backend="iree", device=kind)
            result = run(config, evaluate, generations=generations, key=SEED,
                         backend="iree", device=kind)
            return route, result
        except Exception as error:  # noqa: BLE001 - the device route is an
            # optimisation; the host loop is the floor. The reason is printed,
            # so a declined device route never looks like a success.
            print(f"device route failed ({type(error).__name__}: "
                  f"{str(error).strip().splitlines()[0][:96]})")
    else:
        print("no device (needs the iree packages and a rocm/cuda device)")
    print("falling back to the numpy host loop")
    problem = SymbolicRegression(x, y)
    return "numpy host", run(config, problem.evaluate, generations=generations,
                             key=SEED)


started = time.perf_counter()
route, device_result = run_device(DEVICE_CONFIG, x, y)
wall = time.perf_counter() - started
print(f"route          {route}")
print(f"run            {DEVICE_CONFIG[1].pop_size} trees x "
      f"{device_result.generations} generations in {wall:.2f}s "
      f"(first-call graph compile included)")
print(f"best fitness   {device_result.best_fitness:+.5f}")
host_fitness = SymbolicRegression(x, y).evaluate(device_result.best_channels)
delta = abs(float(host_fitness[0]) - device_result.best_fitness)
print(f"host check     {float(host_fitness[0]):+.5f} for the winning tree "
      f"(the numpy host evaluator agrees to {delta:.2e})")

route          fused kernel
run            128 trees x 5 generations in 11.84s (first-call graph compile included)
best fitness   -78.67564
host check     -78.67565 for the winning tree (the numpy host evaluator agrees to 7.63e-06)


## Where this leaves you

* `example/gp_sr.py` is this walkthrough as a script: module-level knobs, a `main()`, a `report` callback and the same local renderer -- the canonical end-to-end example to copy from.
* `python -m evogp.gp.smoke` is the library's own converging CPU smoke: 200 trees x 100 generations on the harder `x0**4/(x0**4+1) + x1**4/(x1**4+1)` target, in a few seconds.
* `example/gp_custom_loss.py` shows the `CustomLoss` escape hatch (any objective, not just a mean squared error), `example/gp_device.py` is the device section as a script with the timing block, and `example/evox_workflow.py` runs this stack as an `evox` 2.0 `StdWorkflow`.
* Gaps this notebook worked around: no public expression exporter (`MIGRATION.md` step 9) and a single-output stack only (`output_len > 1` raises until step 8 lands).